# Lesson 34 · Nyquist and aliasing — high frequencies in disguise: how 6 kHz passes itself off as 2 kHz

**Today's goal**: understand why "the sample rate must be more than twice the highest frequency", and see for yourself how an undersampled high-frequency sine disguises itself as a low frequency.

> **Closing the loop**: L02 gave you "at least 2 points per cycle"; in L04 a chord at sr=200 tripped us up. This lesson you **predict first, then run**: how 6 kHz passes itself off as 2 kHz.

**Why it matters for Aurora**: a 16 kHz pipeline only "trusts" 0–8 kHz; an anti-aliasing filter cuts off everything above 8 kHz so fake low frequencies don't pollute the mel / MFCC features.

← **Previous lesson** · [L33 · Generating a sine](L33_sine_wave.ipynb)

> Last lesson covered **generating a sine**: x[n]=A·sin(2πfn/sr), implemented yourself and checked against aurora.audio.sine.
> This lesson explores **Nyquist and aliasing**.

## This lesson's story: why did CD audio pick 44100 Hz?

The highest frequency the human ear can hear is about 20 kHz. The CD sample rate is 44100 Hz — just a bit more than twice 20000.

That's no coincidence; it's the **Nyquist theorem**: to sample a signal of frequency `f` without distortion, the sample rate must satisfy `sr > 2f`. Otherwise high frequencies "fold" into a fake low frequency — and that's **aliasing**.

**Intuition**: what happens if you sample a 6 kHz sine wave at 8 kHz?

The samples are `1/8000` seconds apart. In each interval the 6 kHz wave turns `6000/8000 = 0.75` of a turn — more than half a turn, and a digital system "can't tell" 0.75 of a turn from -0.25 of a turn (both produce the same sequence of samples). So 6 kHz is mistaken for `|6000 - 8000·round(6000/8000)| = 2000 Hz`.

### Why do 0.75 of a turn and -0.25 of a turn give the same samples?

It comes down to the periodicity of the sine function. In terms of angle:
- 0.75 of a turn = 0.75 × 2π = 1.5π radians
- -0.25 of a turn = -0.25 × 2π = -0.5π radians

The key trig identity is: **sin(θ) = sin(θ - 2π)** (the sine function has period 2π)

Therefore:
$$\sin(1.5\pi) = \sin(1.5\pi - 2\pi) = \sin(-0.5\pi)$$

Put more concretely: **sin(1.5π) = -1** and **sin(-0.5π) = -1** — both equal -1.

When sampling, the value of a sample where the wave has turned 0.75 of a turn (1.5π) is exactly the same as where it has turned -0.25 of a turn (-0.5π). The system only sees that value and can't tell which angle produced it, so it mistakes the 6 kHz wave, turning 0.75 of a turn, for some low frequency turning -0.25 of a turn. The folding formula tells us that low frequency is 2 kHz.

---

The folding formula:

$$\text{alias}(f, f_s) = \left| f - f_s \cdot \text{round}\!\left(\frac{f}{f_s}\right) \right|$$

Your task in this section: implement `predict_alias_freq(freq, sample_rate)` and verify it on the 8 kHz/6 kHz case.

## 1. The concept

The highest frequency a sample rate `sr` can represent faithfully is `sr/2` (the **Nyquist frequency**). Above it, high frequencies fold into a fake low frequency — and that's **aliasing**.

### Why is the highest frequency sr/2?

Imagine a clock that you glance at once every second.
- If the hand moves 0.4 of a turn (144°) between looks, over 10 looks you see 0°, 144°, 288°, 72°, … — steady forward steps, and you can tell how fast it's going.
- If the hand moves 0.6 of a turn (216°) between looks, you see 0°, 216°, 72°, 288°, … — exactly what a hand turning *backwards* by 144° would show. You can no longer tell its true speed or direction.
- The critical case: **the hand moves exactly 0.5 of a turn between looks**: 0°, 180°, 0°, 180°, … — you can see it flipping, but not which way it's turning. That's the limit of what you can resolve.

In sampling:
- sample interval = 1/sr seconds
- a wave of frequency f turns `f / sr` of a turn in each sample interval
- to resolve the wave you need `f / sr` ≤ 0.5, i.e. **f ≤ sr/2**
- if f > sr/2, the wave turns more than half a turn between adjacent samples, the system "can't see" the true frequency, and aliasing happens.

**Example**: an 8 kHz sample rate → Nyquist = 4 kHz. A 4 kHz wave turns exactly 0.5 of a turn (180°) between samples — the limit of what can be resolved.

---

### Why it folds

Sampling records only the amplitude at discrete instants and loses the continuous detail between samples. A frequency `f` above Nyquist and its mirror frequency produce exactly the same value at every sample, so in the discrete domain the two can't be told apart.

**The folding mechanism**: `round(f/sr)` finds the nearest integer multiple of the sample rate (that is, the nearest folding center `k·sr`), which is used to "pull" the high frequency back into the Nyquist range. `abs(f - k·sr)` computes the distance from the true frequency to that folding center, which gives the apparent frequency.

**Example**: `sr=8000, f=6000`
- `round(6000/8000) = round(0.75) = 1`, so the folding center is `1×8000 = 8000` Hz
- distance `|6000 - 8000| = 2000 Hz`, so 6 kHz sounds like 2 kHz

## Into the lab: take sound apart into arrays you can look at

Generate samples with `sr=8000` and a very short duration (0.01 s), so the arrays are small enough to print directly. The code that follows compares the high-frequency wave at `true_freq=6000` with the folded low-frequency wave and checks that they match point by point at the samples (folding a sine downward also flips its phase, so the 6 kHz samples equal a phase-inverted 2 kHz pure tone).

### Refresher: what does the sine() function look like?

From here on we'll keep calling `sine(freq, duration, sample_rate)` to generate waveforms — the function from L33. If it's been a while and it's a bit hazy, no problem; let's line it up here first:

`sine(freq, duration, sample_rate)` implements exactly this formula:

$$x[n] = \sin\!\left(\frac{2\pi f n}{sr}\right), \quad n = 0, 1, 2, \dots, N-1$$

In other words, it returns an array whose `n`-th element is "the amplitude of a sine wave of frequency `f` at the `n`-th sampling instant". `N` (the array length) is set by `duration × sample_rate`.

This matters a lot: next, this lesson checks whether two sample sequences, the "6 kHz version" and the "2 kHz version", are exactly the same — and if you're not sure what `sine()` actually computes, you can't judge what "matching point by point" means. The code cell below first runs a small example, turning this formula into numbers you can see.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
from aurora.audio import sine

# Refresher from L33: sine(freq, duration, sample_rate) returns x[n] = sin(2*pi*freq*n/sample_rate)
# Use a slower example that's easy to read (1 Hz; the 1000 Hz "sample rate" is just to keep the printout handy here)
demo = sine(freq=1, duration=0.005, sample_rate=1000)
n_demo = np.arange(len(demo))
print('n (which sample)        =', n_demo)
print('sine(1, 0.005, 1000)[n] =', np.round(demo, 3))
print('Compare with the formula x[n] = sin(2π·1·n/1000): computing it point by point should give the same values as above')
print('\nready')

## Hands-on: how a sequence turns into a signal, step by step

Change `sample_rate`, `duration` and `freq`, and watch how the printed length of the time axis (`N`) and the spacing between samples (`1/sr`) change with them.

In [ ]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('sample indices n =', n)
print('time axis t =', np.round(t, 3))
print('angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


## Code experiment: fold every frequency back into the Nyquist range

This experiment loops over frequencies from 0 to 20 Hz, uses `|f - sr·round(f/sr)|` to compute each one's apparent frequency at `sr=10`, and shows how frequencies on either side of Nyquist fold symmetrically.

In [ ]:
import numpy as np

sample_rate = 10
nyquist = sample_rate / 2
freqs = np.arange(0, 21)

for f in freqs:
    alias = abs(((f + nyquist) % sample_rate) - nyquist)
    print(f'true={f:2d}Hz -> alias={alias:4.1f}Hz')


## 2. ✏️ Implement the alias-frequency predictor `predict_alias_freq`

**Check by hand (sr=8000)**:

| True frequency f | round(f/sr) | f - k·sr | alias |
|---|---|---|---|
| 0 | 0 | 0 | **0 Hz** |
| 2000 | 0 | 2000 | **2000 Hz** (below Nyquist, no folding) |
| 4000 | round(0.5)=0 | 4000 | **4000 Hz** (exactly Nyquist) |
| 6000 | 1 | -2000 | **2000 Hz** (folded!) |
| 8000 | 1 | 0 | **0 Hz** (exactly one full turn, aliases to DC) |
| 10000 | 1 | 2000 | **2000 Hz** |

The pattern: frequencies fold as a mirror image about Nyquist (4000 Hz), and the pattern repeats every `sr`.

> **Implementation hint**: `abs(freq - sample_rate * round(freq / sample_rate))`
> Note that Python's `round(0.5)=0` (banker's rounding), which agrees with NumPy's `np.round(0.5)=0.0`.

### What round() means here: finding the "folding center"

What does the `round(f/sr)` in the folding formula do?

**Intuitively**: which "integer multiple of sr" is f closest to?
- `f/sr` tells you how many times sr f is: 0.75 times, 1.2 times, 2.3 times…
- `round(f/sr)` rounds that multiple to the nearest integer k
- that integer k says "which turn of the fold" we're on

**Concrete examples** (sr=8000):
- f=6000 → 6000/8000=0.75 → round(0.75)=1 → turn 1 (folding center 1×8000=8000 Hz)
- f=3000 → 3000/8000=0.375 → round(0.375)=0 → turn 0 (folding center 0×8000=0 Hz)
- f=9000 → 9000/8000=1.125 → round(1.125)=1 → turn 1 (folding center 1×8000=8000 Hz)
- f=10000 → 10000/8000=1.25 → round(1.25)=1 → turn 1 (folding center 1×8000=8000 Hz)

Once you've found the folding center k×sr, use `|f - k×sr|` to compute the distance from f to that center; that distance is "the fake frequency it disguises itself as".

### About Python's round() and banker's rounding

Python uses "banker's rounding":
- 0.75 → 1 ✓ (as you'd expect)
- 0.5 → 0 (rounds to even)
- 1.5 → 2 (rounds to even)
- 2.5 → 2 (rounds to even)

In this lesson's example (6000/8000=0.75) the 0.5 boundary never comes up, so banker's rounding causes no trouble. But if you use NumPy or another library, check that the rounding rules agree. For this lesson's algorithm, **Python's round() and NumPy's np.round() give the same results**.

---

### From "how many turns" to "how many hertz" — filling in the full chain of reasoning

We already know that 6000 Hz sampled at 8000 Hz becomes 2000 Hz, and that `round(0.75)=1` gives a folding center of 8000 Hz. But one step is still missing: **how exactly do you get from the intermediate quantity "0.75 of a turn" to the final answer "2000 Hz"?** Let's lay the whole chain out:

1. **Compute the ratio**: 6000 Hz is `8000 × 0.75`, meaning the wave turns 0.75 of a turn in each sample interval (that's exactly what `f/sr = 6000/8000 = 0.75` means).
2. **Spot the problem**: 0.75 of a turn is past the half-turn (0.5) red line for "can be resolved". The system can only see the values at the samples, and `sin(the angle of 0.75 of a turn) = sin(the angle of -0.25 of a turn)` (we proved this earlier with the periodicity identity), so the system can't tell "turned forward 0.75 of a turn" from "turned backward 0.25 of a turn".
3. **Convert to a frequency**: since 0.75 of a turn looks the same to the system as -0.25 of a turn, convert that "number of turns" back to hertz: `8000 × (-0.25) = -2000` Hz.
4. **Drop the direction**: the minus sign only means "turning backwards"; the ear and the spectrogram only care how fast it turns (the magnitude), not which way, so take the absolute value: `|-2000| = 2000` Hz.

Written out in full, the chain is:

$$6000\text{ Hz} \;\to\; \frac{6000}{8000}=0.75\text{ turn} \;\to\; \text{folds back to } -0.25\text{ turn} \;\to\; 8000\times(-0.25)=-2000\text{ Hz} \;\to\; |-2000|=2000\text{ Hz}$$

This chain and the `round()` formula are two ways of telling the same story: `round(6000/8000)=1` just quickly locates the "folding center" (8000 Hz), while the four steps above explain why the 0.75 of a turn and the -0.25 of a turn on either side of that center really are the same thing — the complete chain of cause and effect behind "why high frequencies pass themselves off as low ones".

### Before writing code, fill in the variable table

Three things to pin down before you write `predict_alias_freq`:
- Input: `freq` (the frequency to test, Hz) and `sample_rate` (the sample rate, Hz)
- Key steps: compute `k = round(freq / sample_rate)`, then `abs(freq - k * sample_rate)`
- Return: a float, the frequency's apparent frequency (in the range 0 to `sample_rate / 2`)

In [ ]:
def predict_alias_freq(freq, sample_rate):
    """Return the apparent frequency of freq when sampled at sample_rate (in the range 0 to Nyquist).
    Folding formula: alias = abs(freq - sample_rate * round(freq / sample_rate))
    """
    # ✏️ TODO: implement the folding formula
    raise NotImplementedError(
        "predict_alias_freq: implement the folding formula abs(freq - sr * round(freq/sr))"
    )


In [ ]:
# Boundary-condition check — run this cell after implementing predict_alias_freq
# These three boundaries are exactly where it's easiest to get the sign or rounding direction wrong
try:
    assert predict_alias_freq(0, 8000) == 0.0,    "freq=0 should return 0.0"
    assert predict_alias_freq(4000, 8000) == 4000.0, "freq=sr/2 should return Nyquist 4000.0"
    assert predict_alias_freq(8000, 8000) == 0.0,  "freq=sr should fold back to 0.0"
    print('✅ All boundary conditions passed')
except (NotImplementedError, TypeError):
    print('⚠️  Implement predict_alias_freq first, then run this cell')


In [ ]:
sr = 8000          # Nyquist = 4000 Hz
true_freq = 6000   # above Nyquist
alias = predict_alias_freq(true_freq, sr)
print('predicted apparent frequency:', alias, 'Hz')
assert 0 <= alias <= sr/2, 'the apparent frequency should fall in 0..Nyquist'

x_aliased = sine(true_freq, duration=0.01, sample_rate=sr)
x_lowfreq = sine(alias,     duration=0.01, sample_rate=sr)

# First, look at the actual sample values
print('\n--- values of the first 10 samples ---')
print(f'n    |  6kHz samples |  2kHz samples | -2kHz samples |')
for n in range(min(10, len(x_aliased))):
    print(f'{n:2d}   |  {x_aliased[n]:8.5f}    |  {x_lowfreq[n]:8.5f}    | {-x_lowfreq[n]:8.5f}   |')

# Folding a sine downward also flips its phase: sin(2π(fs−f)n/fs) = −sin(2πfn/fs)
# so the 6 kHz samples equal the 2 kHz pure tone "negated" — the same 2 kHz frequency, phase-shifted by π.
diff_plus  = float(np.max(np.abs(x_aliased - x_lowfreq)))   # compared with +2kHz
diff_minus = float(np.max(np.abs(x_aliased + x_lowfreq)))   # compared with −2kHz
print(f'\nmax difference from +2kHz: {diff_plus:.2e}')
print(f'max difference from −2kHz: {diff_minus:.2e}')
print('→ the 6 kHz samples = a phase-inverted 2 kHz pure tone (same frequency, phase-shifted by π)')
assert diff_minus < 1e-10, f'aliasing check failed: should match −2kHz point by point, got {diff_minus:.2e}'
print('✅ Aliasing check passed: in the discrete domain the frequencies 6 kHz and 2 kHz cannot be told apart')

## 3. Verifying aliasing: why do the high-frequency samples = the low frequency, phase-inverted?

### Deriving the phase-flip formula

The code shows one key observation: when 6 kHz is sampled at 8 kHz, the sequence of samples equals the 2 kHz low frequency **negated**:

$$\sin\left(2\pi(f_s - f) \frac{n}{f_s}\right) = -\sin\left(2\pi f \frac{n}{f_s}\right)$$

Why does a minus sign appear on its own?

**Derivation** (using the trig identity sin(A - B) = sin A cos B - cos A sin B):

$$\sin\left(2\pi(f_s - f) \frac{n}{f_s}\right) = \sin\left(2\pi n - 2\pi f \frac{n}{f_s}\right)$$

Expanding with sin(A - B):

$$= \sin(2\pi n) \cos\left(2\pi f \frac{n}{f_s}\right) - \cos(2\pi n) \sin\left(2\pi f \frac{n}{f_s}\right)$$

Because n is an integer:
- $\sin(2\pi n) = 0$ (for any integer)
- $\cos(2\pi n) = 1$ (for any integer)

So:

$$= 0 \cdot \cos\left(2\pi f \frac{n}{f_s}\right) - 1 \cdot \sin\left(2\pi f \frac{n}{f_s}\right) = -\sin\left(2\pi f \frac{n}{f_s}\right)$$

**Conclusion**: a high-frequency sine folded downward automatically becomes a phase-inverted version of the low frequency (phase-shifted by π).

---

### Picturing it on the unit circle

Imagine two people walking round the unit circle:
- Red (6 kHz): turns 0.75 of a turn (270°) per sample
- Blue (2 kHz): turns 0.25 of a turn (90°) per sample

Turning +270° lands in the same place as turning −90°, so Red is really walking Blue's path in the opposite direction: at every sample, Red's position is Blue's reflected in the horizontal axis. Their heights (the sin values) are therefore always opposite — the same wave, phase-shifted by 180° = π.

---

## 4. Plot it: the high-frequency samples vs that fake low frequency (a phase-inverted mirror image)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.plot(x_aliased, 'o-', ms=4, label=f'sampled {true_freq} Hz @ {sr} Hz')
plt.plot(x_lowfreq, 'x--', ms=5, label=f'pure {alias:.0f} Hz')
plt.title('Aliasing: a 6 kHz tone masquerades as a low tone')
plt.xlabel('sample n'); plt.legend(); plt.tight_layout(); plt.show()

**Conclusion**: undersampling lets a high frequency disguise itself as a low one. That's why audio goes through an "anti-aliasing filter" before it's recorded.

**🎉 When you're done**: save the notebook (`Ctrl+S`), then commit and push this lesson's changes following the repo conventions.

In [ ]:
# Bonus: compute a few samples by hand to verify 0.75 of a turn = -0.25 of a turn
import numpy as np

sr = 8000
duration = 0.005  # 5 ms, enough to see clearly
n_samples = int(sr * duration)

print("Working it by hand: how sin() computes each sample\n")
print(f"{'n':<3} | {'time t(s)':<10} | {'6kHz turns':<12} | {'sin(6kHz)':<10} | {'2kHz turns':<12} | {'sin(2kHz)':<10} | {'-sin(2kHz)':<10}")
print("-" * 100)

for n in range(min(8, n_samples)):
    t = n / sr

    # phase and sample value of the 6 kHz wave
    phase_6k = 2 * np.pi * 6000 * t
    cycles_6k = 6000 * t
    cycles_6k_wrapped = (cycles_6k % 1)  # fractional part only
    sin_6k = np.sin(phase_6k)

    # phase and sample value of the 2 kHz wave
    phase_2k = 2 * np.pi * 2000 * t
    cycles_2k = 2000 * t
    sin_2k = np.sin(phase_2k)

    # print the results
    print(f"{n:<3} | {t:<10.6f} | {cycles_6k:<12.4f} | {sin_6k:<10.6f} | {cycles_2k:<12.4f} | {sin_2k:<10.6f} | {-sin_2k:<10.6f}")

print("\nWhat to notice:")
print("- the '6kHz turns' column shows how many turns the 6 kHz wave has made (the decimal is a fraction of a turn)")
print("- the '2kHz turns' column shows how many turns the 2 kHz wave has made")
print("- the key observation: the 6 kHz sample values = the 2 kHz sample values negated! (the two right-hand columns are opposite at every point)")
print("- That's aliasing: the 6 kHz sample sequence perfectly impersonates a phase-inverted 2 kHz tone.")

## 🎨 Picture it: aliasing — 6 kHz sampled at 8 kHz disguises itself as a low frequency

In [ ]:
import aviz as aviz; aviz.style()
aviz.aliasing(6000, 8000);

In [ ]:
sr = 12
nyq = sr / 2
for f in [1, 5, 6, 7, 11, 13, 17]:
    alias = abs(((f + nyq) % sr) - nyq)
    status = 'safe' if f <= nyq else 'disguised'
    print(f'{f:>2}Hz -> {alias:>4.1f}Hz  {status}')


## Parameter experiment: sweep 0–32000 Hz and watch the folding pattern

Fix `sr=16000`, sweep `freq` from 0 to 32000, print `predict_alias_freq` every 1000 Hz, and watch the results fold within the 0–8000 Hz range: 16000 Hz folds back to 0 Hz, 24000 Hz folds back to 8000 Hz, and 32000 Hz folds back to 0 Hz again. Paste the code below into an empty cell and run it:

```python
sr = 16000
for f in range(0, 33000, 1000):
    alias = predict_alias_freq(f, sr)
    print(f'{f:>6} Hz  ->  {alias:>6.0f} Hz')
```

The pattern: the folding repeats every `sr` Hz; Nyquist (8000 Hz) is the axis of symmetry, so 9000 Hz and 7000 Hz give the same apparent frequency (both 7000 Hz).

## Before the parameter experiment: the periodicity of folding, and two formulas

### Why does the folding pattern repeat every sr?

Let alias(f, sr) be the folding formula. We want to show: **alias(f + sr, sr) = alias(f, sr)**

**Derivation**:

$$\text{alias}(f + sr) = |f + sr - sr \cdot \text{round}\!\left(\frac{f + sr}{sr}\right)|$$

$$= |f + sr - sr \cdot \left(\text{round}\!\left(\frac{f}{sr}\right) + 1\right)|$$

$$= |f + sr - sr \cdot \text{round}\!\left(\frac{f}{sr}\right) - sr|$$

$$= |f - sr \cdot \text{round}\!\left(\frac{f}{sr}\right)|$$

$$= \text{alias}(f)$$

**Conclusion**: the folding pattern repeats with period sr. That's why the frequencies in the range 0–sr already cover the information of the whole frequency axis.

### Why the two folding formulas are equivalent

In code you may have seen it written two ways:

**Formula 1** (standard form):
$$\text{alias}_1(f) = \left| f - sr \cdot \text{round}\!\left(\frac{f}{sr}\right) \right|$$

**Formula 2** (modulo form):
$$\text{alias}_2(f) = \left| \left(\left(f + \text{nyquist}\right) \bmod sr\right) - \text{nyquist} \right|$$

where nyquist = sr/2.

**Why are they equivalent?**

The core idea of formula 2:
- first "shift" the frequency by nyquist (add nyquist)
- use the modulo to fold it back into the range 0–sr (%sr)
- then "shift back" (subtract nyquist)
- the result lands in the range -nyquist to +nyquist (that is, -sr/2 to +sr/2)

The two formulas are mathematically equivalent, but formula 1 is more intuitive (it finds the nearest folding center directly), while formula 2 is handier for dealing with periodicity. **This lesson uses formula 1**.

---

## Wrap-up

You can now use `predict_alias_freq(freq, sr)` to map any frequency to its apparent frequency in the range 0–Nyquist. The plot confirms it: the samples of the 6 kHz high frequency land on the phase-inverted curve of the 2 kHz low frequency (folding a sine downward flips its phase), so in the discrete domain the frequencies can't be told apart. At a 16 kHz sample rate, Aurora's audio pipeline extracts mel features only for 0–8 kHz; any energy above 8 kHz that isn't cut off by an anti-aliasing filter turns into a fake low frequency and pollutes the model's input. The next lesson (L35, Euler meets the FFT) brings in the complex representation and twiddle factors, laying the groundwork for deriving the DFT; L36 then brings in the Hann window to suppress spectral leakage.

In [ ]:
# Quick check: start from a short sequence and confirm the output of every step
import numpy as np

sample_rate = 8
duration = 1.0
freq = 1.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
x = np.sin(angle)

print('1) What should N be?', N)
print('2) n is the sample index:', n)
print('3) t is the time of each sample in seconds:', np.round(t, 3))
print('4) angle is the angle of each sample on the circle:', np.round(angle, 3))
print('5) x is the final waveform:', np.round(x, 3))


---
⬇️ **Pass check**: once you've read the wrap-up, finish the whiteboard challenge below before ticking the self-review.


## ✏️ Whiteboard challenge: aliasing by hand (target: 10 minutes)

Cover the screen and answer on paper. Hint: use the formula alias = |f - sr·round(f/sr)|, and write out the intermediate value of round() explicitly at every step.

**Question 1**: sr=8000 Hz, f=6000 Hz
- Step 1: compute f/sr = 6000/8000 = ?
- Step 2: round(?) = ? (which integer does it round to?)
- Step 3: sr × round(...) = ?
- Step 4: |6000 - ?| = alias

**Question 2**: sr=8000 Hz, f=3000 Hz (below Nyquist=4000)
- This frequency is below Nyquist, so it should **not fold** and should come back unchanged
- Check with the formula: round(3000/8000) = ?, then work it out

**Question 3**: sr=8000 Hz, f=9000 Hz
- This time f > sr, so it folds past a full turn
- Step 1: f/sr = 9000/8000 = 1.125, round(1.125) = ?
- Carry on from there

**Question 4**: why can a CD (44100 Hz) record up to the 20 kHz limit of human hearing?
- The Nyquist condition: sr > 2·f_max — so what does sr need to exceed?
- 44100/2 = ? Hz (is it above 20000?)
- That's where the CD sample rate comes from

**Question 5**: in Aurora's audio pipeline, if the model's input signal goes up to 8 kHz, what's the minimum sample rate?
- Work backwards from the Nyquist condition

When you've finished deriving, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Question 1: sr=8000, f=6000 → alias=2000
sr = 8000
assert np.isclose(abs(6000 - sr * round(6000/sr)), 2000.0)
try:
    a1 = predict_alias_freq(6000, sr)
    assert np.isclose(a1, 2000.0), f"alias(6000,8000) should be 2000, got {a1}"
    print(f"Q1 ✅  alias(6000, 8000) = {a1} Hz (6 kHz folds to 2 kHz)")
except (NotImplementedError, TypeError):
    print("⬜ Q1: implement predict_alias_freq() first, then run the answer-check cell")

# Question 2: sr=8000, f=3000 → alias=3000 (no folding)
assert np.isclose(abs(3000 - sr * round(3000/sr)), 3000.0)
try:
    a2 = predict_alias_freq(3000, sr)
    assert np.isclose(a2, 3000.0)
    print(f"Q2 ✅  alias(3000, 8000) = {a2} Hz (below Nyquist=4000, no folding)")
except (NotImplementedError, TypeError):
    print("⬜ Q2: implement predict_alias_freq() first, then run the answer-check cell")

# Question 3: sr=8000, f=9000 → alias=1000
assert np.isclose(abs(9000 - sr * round(9000/sr)), 1000.0)
try:
    a3 = predict_alias_freq(9000, sr)
    assert np.isclose(a3, 1000.0)
    print(f"Q3 ✅  alias(9000, 8000) = {a3} Hz (round(9000/8000)=1, |9000-8000|=1000)")
except (NotImplementedError, TypeError):
    print("⬜ Q3: implement predict_alias_freq() first, then run the answer-check cell")

# Question 4: CD at 44100 Hz
nyquist_cd = 44100 / 2
assert nyquist_cd == 22050.0 and nyquist_cd > 20000
print(f"Q4 ✅  44100/2 = {nyquist_cd} Hz > 20000 Hz (the limit of human hearing), Nyquist satisfied")

# Question 5: Aurora's 8 kHz limit → sr > 16000
min_sr = 2 * 8000
print(f"Q5 ✅  minimum sample rate = 2×8000 = {min_sr} Hz; Aurora's usual 16000 Hz sits right at that boundary")
print("\n🎉 Aliasing whiteboard challenge passed! The folding formula |f - sr·round(f/sr)| is now second nature.")

### Going deeper: why doesn't the Nyquist frequency (sr/2) itself fold?

A student asked: "Aliasing is high frequencies folding into low ones — so why doesn't 4000 Hz alias? Why does it still come back as 4000?"

**Answer**: 4000 Hz is exactly the limiting frequency for an 8000 Hz sample rate. It **neither folds nor counts as going past the limit**.

Mathematically:
- f = 4000 Hz, sr = 8000 Hz
- f/sr = 0.5, round(0.5) = 0 (Python's banker's rounding)
- fold center = 0 × 8000 = 0 Hz
- alias = |4000 - 0| = 4000 Hz

So 4000 Hz comes back as itself.

**Why it makes sense conceptually too**:
- a 4 kHz wave turns 0.5 of a turn (exactly 180°) in each sample interval
- every two samples, the wave completes one full cycle
- it's the highest frequency that can be resolved — if the samples happen to land on the peaks, you see the amplitude alternate "max-min-max" (if they land on the zero crossings, as L33 showed, you see nothing but zeros)
- any higher and you can't tell any more (aliasing), but 4 kHz itself doesn't go past the limit

---

In [ ]:
# ✏️ Self-review for this lesson
l34_review = {
    "nyquist_theorem":           None,  # remember the Nyquist condition sr > 2f_max? True/False
    "alias_formula":             None,  # remember alias=|f - sr·round(f/sr)|? True/False
    "predict_alias_implemented": None,  # implemented predict_alias_freq and passed the assertions? True/False
    "cd_sampling_rate":          None,  # understand why 44100 Hz covers the ear's 20 kHz limit? True/False
    "whiteboard_passed":         None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l34_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l34_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L34 fully cleared! On to L35: Euler meets the FFT')

---

→ **Next lesson** · [L35 · Euler meets the FFT](L35_euler_fft.ipynb)

> Next lesson covers **Euler meets the FFT**: what e^{-2πikn/N} is, and twiddle factors visualized.